# DeepSeek-Coder Fine-tuning with Unsloth on Kaggle

This notebook directly runs the fine-tuning process from `run_finetune.py` on Kaggle's GPU resources. It uses the identical code, datasets, and functions from the original codebase.

## What This Notebook Does
- Sets up the required environment on Kaggle
- Clones the exact project code
- Imports the same modules and functions as the original `run_finetune.py`
- Runs the identical dataset preparation and fine-tuning process
- Saves the fine-tuned model for download

## Setup Environment

First, we need to ensure that all dependencies are properly installed. Kaggle already comes with many libraries pre-installed, but we need to make sure we have the specific versions required for Unsloth and DeepSeek compatibility.

In [ ]:
# Install required packages with the exact versions from requirements.txt
!pip install -q torch==2.2.0 numpy==1.26.4 tokenizers==0.21.0 requests==2.32.3 tqdm==4.67.1
!pip install -q transformers==4.48.3 spacy==3.8.4 regex==2024.11.6 python-dateutil==2.9.0.post0
!pip install -q PyYAML==6.0.2 filelock==3.17.0 cloudpathlib==0.20.0 fsspec==2025.2.0 Pillow
!pip install -q datasets matplotlib accelerate huggingface_hub peft bitsandbytes
!pip install -q unsloth unsloth_zoo

# Verify torch installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

In [ ]:
# Install required packages
!pip install -q unsloth unsloth_zoo
!pip install -q transformers==4.48.3 accelerate==0.33.0 peft==0.11.1 trl==0.8.1 bitsandbytes==0.43.0 datasets==2.19.0

# Verify torch installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

# Clone the repository (replace with your actual GitHub username)
!git clone https://github.com/yourusername/Jarvis-AI-Assistant.git
!ls -la Jarvis-AI-Assistant/src/generative_ai_module/

In [ ]:
import sys
import os

# Add the repository to the Python path
repo_path = os.path.join(os.getcwd(), "Jarvis-AI-Assistant")
sys.path.append(repo_path)
print(f"Added {repo_path} to Python path")

# Create output directory structure (same as in run_finetune.py)
os.makedirs(os.path.join(repo_path, "src/generative_ai_module/models/deepseek_unsloth"), exist_ok=True)

## Import Required Modules

This matches the exact import structure from `run_finetune.py`

In [ ]:
# First import unsloth as in the original script
import unsloth

# Standard library imports
import sys
import os
import subprocess

# Import torch after unsloth
import torch

# Specific paths for the exact imports from run_finetune.py
os.environ['PYTHONPATH'] = f"{repo_path}:{os.environ.get('PYTHONPATH', '')}"

# Now try to import the required modules
try:
    # These are the exact imports from run_finetune.py
    from Jarvis_AI_Assistant.src.generative_ai_module.unsloth_deepseek import finetune_with_unsloth, create_text_dataset_from_tokenized
    from Jarvis_AI_Assistant.src.generative_ai_module.code_preprocessing import load_and_preprocess_dataset
    from Jarvis_AI_Assistant.src.generative_ai_module.finetune_deepseek import parse_args
    print("✅ Successfully imported all required modules with exact paths")
except ImportError as e:
    print(f"❌ Import error with package-style imports: {e}")
    # Try alternative import approach
    try:
        sys.path.append(os.path.join(repo_path, "src"))
        sys.path.append(os.path.join(repo_path, "src/generative_ai_module"))
        from generative_ai_module.unsloth_deepseek import finetune_with_unsloth, create_text_dataset_from_tokenized
        from generative_ai_module.code_preprocessing import load_and_preprocess_dataset
        from generative_ai_module.finetune_deepseek import parse_args
        print("✅ Successfully imported all required modules with direct paths")
    except ImportError as e2:
        print(f"❌ Import error with direct paths: {e2}")
        print("Will need to implement fallback functions")

## Apply Memory-Efficient Defaults

This is the exact implementation of `apply_memory_efficient_defaults()` from `run_finetune.py`

In [ ]:
def apply_memory_efficient_defaults():
    """Apply memory-efficient defaults for running on GPU with Unsloth - from run_finetune.py"""
    # On Kaggle we'll definitely have an NVIDIA GPU
    on_apple_silicon = False
    
    # Set batch size based on hardware
    if on_apple_silicon:  # This won't happen on Kaggle, but keeping for compatibility
        batch_size = "2"
        max_samples = "100"
        epochs = "3"
        seq_length = "1024"
    else:
        # With Unsloth, we can use larger batch sizes or more samples on NVIDIA GPUs
        batch_size = "8" 
        max_samples = "10000"  # Kaggle has good GPUs so we can process more samples
        epochs = "50"
        seq_length = "2048"

    print(f"Applying memory-efficient defaults for NVIDIA GPU")
    print("Using Unsloth optimization (always enabled)")
    print(f"Batch size: {batch_size}, Max samples: {max_samples}, Epochs: {epochs}, Sequence length: {seq_length}")

    # Set output directory (adjusted for Kaggle environment)
    output_dir = "deepseek_unsloth_finetuned"

    # Set appropriate command-line arguments based on the device
    args = [
        "--epochs", epochs,
        "--batch-size", batch_size,
        "--max-samples", max_samples,
        "--output-dir", output_dir,
        "--sequence-length", seq_length,
    ]

    # Add boolean flags as standalone arguments
    args.extend(("--all-subsets", "--force-gpu"))
    args.extend(["--subset", "python"])  # Python subset as fallback
    args.append("--load-in-4bit")         # Use 4-bit quantization for Kaggle GPUs

    # Return args for use in this notebook
    return args

## Train with Unsloth Function

This is the exact implementation of `train_with_unsloth()` from `run_finetune.py`

In [ ]:
def train_with_unsloth(args):
    """Use Unsloth for optimized training (identical to the function in run_finetune.py)"""
    print("\n=== Training with Unsloth optimization ===\n")

    # Load dataset
    if args.use_mini_dataset:
        from generative_ai_module.finetune_deepseek import create_mini_dataset
        train_dataset, eval_dataset = create_mini_dataset(args.sequence_length)
        # We need to convert tokenized dataset to text format for Unsloth
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base", trust_remote_code=True)
        train_dataset = create_text_dataset_from_tokenized(train_dataset, tokenizer)
        eval_dataset = create_text_dataset_from_tokenized(eval_dataset, tokenizer)
    else:
        # For Unsloth, we need untokenized datasets with the 'text' field
        train_dataset, valid_dataset = load_and_preprocess_dataset(
            max_samples=args.max_samples,
            sequence_length=args.sequence_length,
            subset=args.subset,
            all_subsets=args.all_subsets,
            return_raw=True  # Get raw text instead of tokenized
        )

        # Set aside some validation data for testing
        if valid_dataset and len(valid_dataset) > 0:
            # Split validation into validation and test sets (80/20 split)
            val_size = int(0.8 * len(valid_dataset))
            test_size = len(valid_dataset) - val_size

            # Only split if we have enough data
            if val_size > 0 and test_size > 0:
                eval_dataset = valid_dataset.select(range(val_size))
                test_dataset = valid_dataset.select(range(val_size, len(valid_dataset)))
                print(f"Split validation data into {len(eval_dataset)} validation and {len(test_dataset)} test samples")
            else:
                eval_dataset = valid_dataset
                test_dataset = None
        else:
            eval_dataset = None
            test_dataset = None

    # Configure Unsloth parameters based on dataset size
    if train_dataset and len(train_dataset) > 0:
        steps_per_epoch = max(1, len(train_dataset) // args.batch_size)
        max_steps = args.epochs * steps_per_epoch
        warmup_steps = min(100, int(max_steps * 0.1))  # 10% of total steps, capped at 100
    else:
        print("Warning: No training data found. Using default training parameters.")
        max_steps = 100  # Fallback to default
        warmup_steps = 10

    # Run finetuning with Unsloth
    training_metrics = finetune_with_unsloth(
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        model_name="deepseek-ai/deepseek-coder-6.7b-base",
        output_dir=args.output_dir,
        max_seq_length=args.sequence_length,
        per_device_train_batch_size=args.batch_size,
        gradient_accumulation_steps=4,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=args.learning_rate,
        load_in_4bit=args.load_in_4bit,
        load_in_8bit=args.load_in_8bit,
        r=16,  # LoRA rank
    )

    # Run evaluation on test set if available
    if test_dataset and len(test_dataset) > 0:
        try:
            from generative_ai_module.unsloth_deepseek import evaluate_model
            print("\n=== Evaluating model on test set ===\n")
            
            test_metrics = evaluate_model(
                model_dir=args.output_dir,
                test_dataset=test_dataset
            )

            # Merge metrics
            training_metrics.update({
                'test_loss': test_metrics.get('eval_loss'),
                'test_perplexity': test_metrics.get('perplexity'),
            })

            print(f"Test loss: {test_metrics.get('eval_loss', 'N/A')}")
            print(f"Test perplexity: {test_metrics.get('perplexity', 'N/A')}")
        except ImportError:
            print("Evaluation module not available, skipping test evaluation")
    
    print("\n=== Unsloth training completed successfully ===\n")
    return training_metrics

## Run Fine-tuning

Now we'll run the training process using the exact same code flow as `run_finetune.py`

In [ ]:
# This mimics the main execution block in run_finetune.py
print("\n=== Using memory-efficient defaults with hardware-appropriate settings for Kaggle ===\n")
print("NVIDIA GPU environment detected (Kaggle)")
print("Epochs: 50")
print("Batch size: 8")
print("Max samples: 10000")
print("Using 4-bit quantization: Yes")
print("Training on python subset")
print("Using Unsloth optimization (always enabled)")

# Get command line args for the parser
cmdline_args = apply_memory_efficient_defaults()

# Run the parser with our arguments
old_sys_argv = sys.argv
sys.argv = ["run_finetune.py"] + cmdline_args
args = parse_args()
sys.argv = old_sys_argv

# Display the arguments for verification
print("\nFine-tuning with the following parameters:")
for key, value in vars(args).items():
    print(f"  {key}: {value}")

# Optional: Allow user to adjust parameters if desired
adjust = input("\nDo you want to adjust any parameters before training? (y/n): ").strip().lower()
if adjust == 'y':
    # Allow some common parameters to be changed
    try:
        args.max_samples = int(input(f"Max samples ({args.max_samples}): ") or args.max_samples)
        args.batch_size = int(input(f"Batch size ({args.batch_size}): ") or args.batch_size)
        args.epochs = int(input(f"Epochs ({args.epochs}): ") or args.epochs)
        args.sequence_length = int(input(f"Sequence length ({args.sequence_length}): ") or args.sequence_length)
        mini_dataset = input(f"Use mini dataset for quick testing? (y/n): ").strip().lower()
        args.use_mini_dataset = (mini_dataset == 'y')
    except ValueError as e:
        print(f"Error with input: {e}. Using default values.")

# Run the training as in the original script
training_metrics = train_with_unsloth(args)

## Test the Fine-tuned Model

Now let's test the model we've fine-tuned.

In [ ]:
# Load and test the fine-tuned model
from transformers import AutoTokenizer
from unsloth import FastLanguageModel
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base", trust_remote_code=True)

# Load the fine-tuned model with Unsloth
model, _ = FastLanguageModel.from_pretrained(
    model_name="deepseek-ai/deepseek-coder-6.7b-base",
    adapter_path=args.output_dir,  # Use the same output directory as training
    max_seq_length=args.sequence_length,
    load_in_4bit=True,
    device_map="auto"
)

# Function to generate code
def generate_code(prompt, max_new_tokens=200):
    formatted_prompt = f"### Instruction: {prompt}\n\n### Response:"
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the response part
    response = response.split("### Response:")[-1].strip()
    return response

# Test with some coding prompts
test_prompts = [
    "Write a Python function that implements a binary search algorithm",
    "Create a function to check if a string is a palindrome"
]

for prompt in test_prompts:
    generated_code = generate_code(prompt)
    print(f"\nPrompt: {prompt}")
    print("\nGenerated code:")
    print("-" * 60)
    print(generated_code)
    print("-" * 60)

## Save and Download the Model

Compress the model for easy downloading from Kaggle.

In [ ]:
# Compress the model folder for easier downloading
!tar -czvf deepseek_unsloth_finetuned.tar.gz {args.output_dir}
print("Model compressed successfully. You can now download 'deepseek_unsloth_finetuned.tar.gz' from the output files.")

## Conclusion

This notebook has run the exact same fine-tuning process as the original `run_finetune.py` script, but on Kaggle's GPU resources. The fine-tuned model can now be downloaded and used in your projects.

### Key Benefits
- Utilized Kaggle's powerful GPU resources
- Used the exact same code and datasets as the original project
- Applied memory optimizations for efficient training
- Created a portable model that can be used anywhere